# Phase 2 — LoRA-QAT (INT8 & INT4)

**Goal:** measure the cheapest QAT variant. Quantize the base model first, then attach small LoRA adapters (rank 16, ~1–2% of total parameters) to recover quality. Base weights stay frozen and quantized; only the adapters train.

**Why this matters:** if LoRA-QAT lands close to full Scheduled QAT, the cost-quality tradeoff favours it for production deployments where you can't afford a full QAT run per model variant.

**Inputs:** `/kaggle/input/sqat-baseline/results/baseline/fp32_logits.pt`

**Outputs:**
- `models/checkpoints/lora_qat_int{4,8}/final_adapter/` (PEFT adapter dirs)
- `results/lora_qat_int{4,8}/training_results.json`

**Estimated time on T4:** ~2.5 h total (fewer trainable params → fewer optimizer FLOPs per step).

## 1. Setup

In [ ]:
import os, sys, shutil, subprocess
from pathlib import Path

WORKING_DIR  = "/kaggle/working"
REPO_DIR     = f"{WORKING_DIR}/scheduled-qat-for-slm"
GITHUB_URL   = "https://github.com/JpCurada/scheduled-qat-for-slm.git"
REPO_DATASET = "/kaggle/input/sqat-repo"
BASELINE_DIR = "/kaggle/input/sqat-baseline/results/baseline"

if not os.path.exists(REPO_DIR):
    if os.path.exists(REPO_DATASET):
        shutil.copytree(REPO_DATASET, REPO_DIR)
    else:
        subprocess.run(["git", "clone", "--depth", "1", GITHUB_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

FP32_LOGITS = Path(BASELINE_DIR) / "fp32_logits.pt"
assert FP32_LOGITS.exists(), "FP32 logits not found — add sqat-baseline as input"

In [ ]:
# peft is needed for LoRA adapters; viz for plots.
!pip install -q -e ".[viz]" peft
import torch
print(f"GPU: {torch.cuda.get_device_name() if torch.cuda.is_available() else 'CPU'}")

## 2. Kaggle config overrides

LoRA-QAT uses `epochs=2` and a higher LR (2e-4) in the published config because only the adapters train. We keep those choices and just shrink seq_length / batch / accum to fit Kaggle.

In [ ]:
import yaml

MODEL_CACHE = f"{WORKING_DIR}/models/base"
KAGGLE_CFG  = Path(REPO_DIR) / "configs_kaggle" / "lora_qat"
KAGGLE_CFG.mkdir(parents=True, exist_ok=True)

def patch_lora_config(src: str, dst: Path) -> Path:
    with open(src) as f:
        cfg = yaml.safe_load(f)
    cfg["model"]["cache_dir"] = MODEL_CACHE
    cfg["data"]["seq_length"] = 512
    cfg["training"]["epochs"] = 1               # 1 epoch is enough at LoRA's higher LR
    cfg["training"]["batch_size"] = 4
    cfg["training"]["gradient_accumulation_steps"] = 8
    cfg["training"]["warmup_steps"] = 50
    cfg["logging"]["save_every_steps"] = 999999
    cfg["logging"]["eval_every_steps"] = 500
    cfg["logging"]["log_dir"] = f"{WORKING_DIR}/results/logs/{dst.stem}/"
    cfg["export"]["output_dir"] = f"{WORKING_DIR}/models/gguf/"
    with dst.open("w") as f:
        yaml.safe_dump(cfg, f, sort_keys=False)
    return dst

lora8_cfg = patch_lora_config("configs/lora_qat/lora_qat_int8.yaml", KAGGLE_CFG / "lora_qat_int8.yaml")
lora4_cfg = patch_lora_config("configs/lora_qat/lora_qat_int4.yaml", KAGGLE_CFG / "lora_qat_int4.yaml")

print("INT8 config:", lora8_cfg)
print("INT4 config:", lora4_cfg)

## 3. LoRA-QAT — INT8

Frozen quantized base + trainable rank-16 LoRA adapters on q/k/v/o projections.

In [ ]:
import gc
from src.training.trainer import run_experiment

results_int8 = run_experiment(
    config_path=str(lora8_cfg),
    device_str="cuda",
    baseline_logits=str(FP32_LOGITS),
    run_benchmarks=False,
)

print("\nLoRA-QAT INT8:")
for k, v in results_int8.items():
    print(f"  {k:25s}  {v}")

gc.collect(); torch.cuda.empty_cache()

## 4. LoRA-QAT — INT4

Same setup, INT4 base. The adapters compensate for the larger quantization error.

In [ ]:
results_int4 = run_experiment(
    config_path=str(lora4_cfg),
    device_str="cuda",
    baseline_logits=str(FP32_LOGITS),
    run_benchmarks=False,
)

print("\nLoRA-QAT INT4:")
for k, v in results_int4.items():
    print(f"  {k:25s}  {v}")

gc.collect(); torch.cuda.empty_cache()

## 5. Adapter parameter count

Quick sanity check on how many parameters were actually trained. For SmolLM2-1.7B with rank 16 on q/k/v/o, expect on the order of 4M trainable params (~0.25% of total). The exact number depends on hidden_size and num_attention_heads.

In [ ]:
# Reload the INT4 adapter and report trainable param count.
from peft import PeftModel
from transformers import AutoModelForCausalLM

adapter_dir = Path(WORKING_DIR) / "models" / "checkpoints" / "lora_qat_int4" / "final_adapter"
if adapter_dir.exists():
    base = AutoModelForCausalLM.from_pretrained(
        "HuggingFaceTB/SmolLM2-1.7B", cache_dir=MODEL_CACHE, torch_dtype=torch.float16,
    )
    peft_model = PeftModel.from_pretrained(base, str(adapter_dir))
    total = sum(p.numel() for p in peft_model.parameters())
    trainable = sum(p.numel() for p in peft_model.parameters() if p.requires_grad)
    print(f"Total params      : {total:,}")
    print(f"Trainable (LoRA)  : {trainable:,}")
    print(f"Trainable fraction: {trainable / total * 100:.3f}%")
    del peft_model, base
    torch.cuda.empty_cache()
else:
    print(f"Adapter dir not found: {adapter_dir}")

## 6. Comparison table

In [ ]:
import json
import pandas as pd

with open(Path(BASELINE_DIR) / "baseline_results.json") as f:
    fp32 = json.load(f)

rows = [
    {"method": "FP32",     "bits": 32, "perplexity": fp32["perplexity"], "kl_divergence": 0.0},
    {"method": "LoRA-QAT", "bits": 8,  "perplexity": results_int8.get("perplexity"), "kl_divergence": results_int8.get("kl_divergence")},
    {"method": "LoRA-QAT", "bits": 4,  "perplexity": results_int4.get("perplexity"), "kl_divergence": results_int4.get("kl_divergence")},
]
df = pd.DataFrame(rows)
df["ppl_delta_pct"] = ((df["perplexity"] / df["perplexity"].iloc[0]) - 1.0) * 100
df.round(4)

In [ ]:
summary_path = Path(WORKING_DIR) / "results" / "lora_qat_summary.json"
summary_path.parent.mkdir(parents=True, exist_ok=True)
with summary_path.open("w") as f:
    json.dump({"int8": results_int8, "int4": results_int4, "fp32": fp32},
              f, indent=2, default=str)
print(f"Wrote {summary_path}")

## 7. Training loss curves

In [ ]:
import json
import plotly.graph_objects as go
from plotly.subplots import make_subplots

def read_jsonl(path):
    if not Path(path).exists():
        return []
    with open(path) as f:
        return [json.loads(l) for l in f if l.strip()]

log8 = read_jsonl(f"{WORKING_DIR}/results/logs/lora_qat_int8/training_steps.jsonl")
log4 = read_jsonl(f"{WORKING_DIR}/results/logs/lora_qat_int4/training_steps.jsonl")

fig = make_subplots(rows=1, cols=2, subplot_titles=("INT8", "INT4"),
                    specs=[[{"secondary_y": True}, {"secondary_y": True}]])
for col, (log, title) in enumerate([(log8, "INT8"), (log4, "INT4")], 1):
    if not log:
        fig.add_annotation(text="no log entries", row=1, col=col,
                           x=0.5, y=0.5, showarrow=False, xref="paper", yref="paper")
        continue
    steps = [e["step"] for e in log]
    fig.add_trace(go.Scatter(x=steps, y=[e["loss"] for e in log], name="train loss",
                             line=dict(color="#4C72B0"), legendgroup=str(col)),
                  row=1, col=col, secondary_y=False)
    val = [e.get("val_ppl") for e in log]
    if any(v is not None for v in val):
        fig.add_trace(go.Scatter(x=steps, y=val, name="val ppl",
                                 line=dict(color="#DD8452", dash="dash"),
                                 legendgroup=str(col)),
                      row=1, col=col, secondary_y=True)

fig.update_xaxes(title_text="step")
fig.update_yaxes(title_text="loss", secondary_y=False, row=1, col=1)
fig.update_yaxes(title_text="val ppl", secondary_y=True, row=1, col=2)
fig.update_layout(title="LoRA-QAT — training curves",
                  height=420, hovermode="x unified",
                  margin=dict(t=80, b=40, l=60, r=60))
fig.show()

## 8. Plotly comparison — quality vs FP32

In [ ]:
def plot_method_comparison(rows, title, fp32_ppl=None):
    labels = [r["label"] for r in rows]
    fig = make_subplots(rows=1, cols=3,
                        subplot_titles=("Perplexity (lower=better)",
                                        "KL Divergence vs FP32 (lower=better)",
                                        "Model size (GB)"))
    fig.add_trace(go.Bar(x=labels, y=[r["perplexity"] for r in rows],
                         marker_color="#4C72B0",
                         text=[f"{r['perplexity']:.3f}" if r["perplexity"] is not None else "—" for r in rows],
                         textposition="outside"), 1, 1)
    if fp32_ppl is not None:
        fig.add_hline(y=fp32_ppl, line_dash="dash", line_color="black",
                      annotation_text=f"FP32 = {fp32_ppl:.3f}", row=1, col=1)
    fig.add_trace(go.Bar(x=labels, y=[r["kl_divergence"] for r in rows],
                         marker_color="#DD8452",
                         text=[f"{r['kl_divergence']:.4f}" if r["kl_divergence"] is not None else "—" for r in rows],
                         textposition="outside"), 1, 2)
    fig.add_trace(go.Bar(x=labels, y=[r["size_gb"] for r in rows],
                         marker_color="#55A868",
                         text=[f"{r['size_gb']:.2f}" for r in rows],
                         textposition="outside"), 1, 3)
    fig.update_layout(title=title, showlegend=False, height=420,
                      margin=dict(t=80, b=40, l=40, r=20))
    fig.show()

rows = [
    {"label": "FP32 baseline", "perplexity": fp32.get("perplexity"),         "kl_divergence": 0.0,                                 "size_gb": 6.5},
    {"label": "LoRA-QAT INT8", "perplexity": results_int8.get("perplexity"), "kl_divergence": results_int8.get("kl_divergence"),   "size_gb": 1.75},
    {"label": "LoRA-QAT INT4", "perplexity": results_int4.get("perplexity"), "kl_divergence": results_int4.get("kl_divergence"),   "size_gb": 0.87},
]
plot_method_comparison(rows, "LoRA-QAT — quantization quality vs FP32",
                       fp32_ppl=fp32.get("perplexity"))

## 9. Sample inference

Same prompts and helper as notebooks 02–04 — direct apples-to-apples comparison across notebooks.

LoRA-QAT inference reload uses `load_lora_checkpoint(base_model_name, adapter_dir, device, cache_dir)` which loads the FP32 base from cache and then applies the trained adapter on top.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from src.quantization.lora_qat import load_lora_checkpoint

SAMPLE_PROMPTS = [
    "The capital of France is",
    "Python is a programming language used for",
    "Once upon a time, in a small village,",
    "The chemical symbol for gold is",
    "To improve your sleep quality, you should",
]
MAX_NEW_TOKENS = 30
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM2-1.7B", cache_dir=MODEL_CACHE)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def generate_with_model(model, prompts=SAMPLE_PROMPTS, max_new_tokens=MAX_NEW_TOKENS):
    model.eval()
    out = []
    for p in prompts:
        ids = tokenizer(p, return_tensors="pt").input_ids.to(device)
        with torch.no_grad():
            gen = model.generate(
                ids, max_new_tokens=max_new_tokens, do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
            )
        out.append(tokenizer.decode(gen[0][ids.shape[1]:], skip_special_tokens=True).strip())
    return out

def free():
    gc.collect(); torch.cuda.empty_cache()

# 1. FP32 baseline
print("Generating FP32 ...")
fp32_model = AutoModelForCausalLM.from_pretrained(
    "HuggingFaceTB/SmolLM2-1.7B", cache_dir=MODEL_CACHE, torch_dtype=torch.float16,
).to(device)
fp32_outs = generate_with_model(fp32_model)
del fp32_model; free()

def load_lora_for_inference(adapter_dir):
    if not Path(adapter_dir).exists():
        print(f"  WARNING — adapter dir missing: {adapter_dir}")
        return None
    return load_lora_checkpoint(
        base_model_name="HuggingFaceTB/SmolLM2-1.7B",
        adapter_dir=adapter_dir,
        device=device,
        cache_dir=MODEL_CACHE,
    )

# 2. LoRA-QAT INT8
print("Generating LoRA-QAT INT8 ...")
m8 = load_lora_for_inference(f"{WORKING_DIR}/models/checkpoints/lora_qat_int8/final_adapter")
int8_outs = generate_with_model(m8) if m8 is not None else ["[adapter missing]"] * len(SAMPLE_PROMPTS)
del m8; free()

# 3. LoRA-QAT INT4
print("Generating LoRA-QAT INT4 ...")
m4 = load_lora_for_inference(f"{WORKING_DIR}/models/checkpoints/lora_qat_int4/final_adapter")
int4_outs = generate_with_model(m4) if m4 is not None else ["[adapter missing]"] * len(SAMPLE_PROMPTS)
del m4; free()

print("Done.")

In [ ]:
import pandas as pd
from IPython.display import display, HTML

inference_df = pd.DataFrame({
    "Prompt":          SAMPLE_PROMPTS,
    "FP32":            fp32_outs,
    "LoRA-QAT INT8":   int8_outs,
    "LoRA-QAT INT4":   int4_outs,
})

inference_df.to_json(Path(WORKING_DIR) / "results" / "lora_qat_inference.json",
                     orient="records", indent=2)

display(HTML(inference_df.to_html(index=False, escape=False).replace(
    "<table", '<table style="font-family:monospace; font-size:12px"')))

## Next steps

Save outputs as `sqat-lora-qat`. Proceed to `06_export_gguf.ipynb` for edge deployment formats.